In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [38]:
def guide_text_gen(text):

    annot_name = []
    prev_text = ""

    for i in text.split(";")[:]:

        sub_text = i.split("__")[1]
        if sub_text == "":  # or sub_text.startswith("CAG"):
            continue
        # finalized_text = sub_text.replace(" ", "-")
        # finalized_text = finalized_text.split("_")[0]

        # finalized_text = sub_text.split("_")[0]

        finalized_text = sub_text

        if finalized_text == prev_text:
            continue
        annot_name.append(finalized_text)

        prev_text = finalized_text

    return ".".join(annot_name)

### Use ML feature importance data

In [39]:
featsImp = pd.read_csv("../results/clr_featImp.csv", index_col=0)
# featsImp = pd.read_csv("../results/clr_featImp_revised.csv", index_col=0)

featsImp_cname = pd.read_csv("../results/clr_featImp_cname.csv", index_col=0)

for col in [
    "dairy_products_meat_fish_eggs_tofu",
    "vegetables_fruits",
    "sweets_salty_snacks_alcohol",
    "non_alcoholic_beverages",
    "grains_potatoes_pulses",
    "oils_fats_nuts",
]:
    del featsImp[col]


featsImp = featsImp.merge(featsImp_cname, left_index=True, right_index=True, how="left")
featsImp.to_csv("../results/clr_featImp_revised.csv")

In [5]:
select_target_cols = [
    "HEI",
    "vegetables_fruits",
    "oils_nuts_fg_eaten",
    "coffee_fg_eaten",
    "meat_fg_eaten",
]
featsImp = featsImp[select_target_cols]

featsImp.sort_values("HEI", inplace=True, ascending=False)

featsImp.head()

/var/folders/kl/wv2_2q3n4z3d2bxtztx97rgc0000gn/T/ipykernel_65847/3156795886.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  featsImp.sort_values("HEI", inplace=True, ascending=False)


,HEI,vegetables_fruits,oils_nuts_fg_eaten,coffee_fg_eaten,meat_fg_eaten
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__Eubacterium_J;s__Eubacterium_J plexicaudatum,0.005489,0.003418,0.007295,NaN,NaN
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__Marvinbryantia;s__Marvinbryantia sp900066075,0.005333,0.004853,0.003426,NaN,NaN
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__CAG-194;s__CAG-194 sp000432915,0.005214,0.006550,0.003390,NaN,NaN
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__TANB77;f__CAG-508;g__CAG-269;s__CAG-269 sp000431335,0.004808,0.004600,0.003242,NaN,NaN
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__Lachnospira;s__,0.004003,0.003665,NaN,NaN,NaN


In [6]:
#### Feature importance processing

print("shape before processing: ", featsImp.shape)

## fill NA with 0
featsImp = featsImp.fillna(0)

## scale the values (min-max scaling)
featsImp = (featsImp - featsImp.min()) / (featsImp.max() - featsImp.min())

## remove rows with all 0 values
featsImp = featsImp[~(featsImp == 0).all(axis=1)]

print("shape after processing: ", featsImp.shape)

shape before processing:  (336, 5)
shape after processing:  (65, 5)


In [7]:
## removing singlets to reduce clades
featsImp = featsImp[~featsImp.index.str.contains("Peptococcia")]

### Adding Taxonomy

In [8]:
taxonomy = pd.read_csv(
    "../../../qiime/taxonomy_rarefied-table_2022_10/taxonomy.tsv", sep="\t"
)
taxonomy["Taxon"] = taxonomy["Taxon"].map(lambda i: i.replace("; ", ";"))
taxonomy = taxonomy.set_index("Feature ID")["Taxon"].to_dict()

#### Adding Microbe Prevalence Data

In [9]:
prevalence_microbes = pd.read_csv("../../../data/prevalence_microbes.csv", index_col=0)
prevalence_microbes.columns = ["prevalence"]

prevalence_microbes["taxonomy"] = prevalence_microbes.index.map(taxonomy)

prevalence_microbes.head()

,prevalence,taxonomy
#OTU ID,,
8937656c16c20701c107e715bad86732,0.858871,d__Bacteria;p__Firmicutes_A;c__Clostridia_2584...
59196a586276f0be745d0e334fc071c6,0.983871,d__Bacteria;p__Firmicutes_A;c__Clostridia_2584...
ad52a0f6646c574417ce0b13b84acfa9,0.740927,d__Bacteria;p__Firmicutes_A;c__Clostridia_2584...
877f27b47c85f6b2d3cf8a6aa86def40,0.796371,d__Bacteria;p__Firmicutes_A;c__Clostridia_2584...
6c7fa77831ae630967a6fbfc8ee47901,0.986895,d__Bacteria;p__Firmicutes_A;c__Clostridia_2584...


In [10]:
def assign_prevalence(microbe, prevalence_microbes):
    taxonomy_match = prevalence_microbes[prevalence_microbes["taxonomy"] == microbe]
    if len(taxonomy_match) > 0:
        taxonomy_match = taxonomy_match.sort_values("prevalence", ascending=False)
        return taxonomy_match.iloc[0]["prevalence"]
    else:
        return 0


featsImp["prevalence"] = featsImp.index.map(
    lambda i: assign_prevalence(i, prevalence_microbes)
)

In [11]:
featsImp.head()

,HEI,vegetables_fruits,oils_nuts_fg_eaten,coffee_fg_eaten,meat_fg_eaten,prevalence
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__Eubacterium_J;s__Eubacterium_J plexicaudatum,1.000000,0.521874,1.000000,0.0,0.0,0.774194
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__Marvinbryantia;s__Marvinbryantia sp900066075,0.971545,0.740868,0.469643,0.0,0.0,0.438508
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__CAG-194;s__CAG-194 sp000432915,0.949909,1.000000,0.464615,0.0,0.0,0.206653
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__TANB77;f__CAG-508;g__CAG-269;s__CAG-269 sp000431335,0.875847,0.702322,0.444458,0.0,0.0,0.350806
d__Bacteria;p__Firmicutes_A;c__Clostridia_258483;o__Lachnospirales;f__Lachnospiraceae;g__Lachnospira;s__,0.729263,0.559517,0.000000,0.0,0.0,0.615927


In [12]:
featsImp.index.nunique()

64

#### Preparing Label Guide Text

In [13]:
featsImp["guide_text"] = featsImp.index.map(guide_text_gen)

In [14]:
with open("nutrition_features_tree.txt", "w") as f:
    for item in featsImp["guide_text"].values:
        f.write("%s\n" % item)

In [15]:
featsImp.index.map(
    lambda i: i.split(";")[1].split("__")[1].split("_")[0]
).value_counts()

Firmicutes           56
Actinobacteriota      3
Bacteroidota          2
Verrucomicrobiota     2
Desulfobacterota      1
dtype: int64

In [16]:
featsImp.index.map(lambda i: i.split(";")[2].split("__")[1]).value_counts()

Clostridia_258483    50
Bacilli               5
Coriobacteriia        2
Bacteroidia           2
Actinomycetia         1
Desulfovibrionia      1
Negativicutes         1
Lentisphaeria         1
Verrucomicrobiae      1
dtype: int64

In [17]:
featsImp.index.map(
    lambda i: i.split(";")[4].split("__")[1].split("_")[0]
).value_counts()[:10]

Lachnospiraceae       24
Acutalibacteraceae     6
Ruminococcaceae        5
Oscillospiraceae       4
Clostridiaceae         2
Butyricicoccaceae      2
Coprobacillaceae       2
Bacteroidaceae         2
Eggerthellaceae        2
Dialisteraceae         1
dtype: int64

#### Annotation Preparation

In [18]:
# settings = {
#     "clade_separation": 0.5,
#     "branch_thickness": 1.5,
#     "branch_bracket_depth": 0.8,
#     "branch_bracket_width": 0.25,
#     "clade_marker_size": 40,
#     "clade_marker_edge_color": "#555555",
#     "clade_marker_edge_width": 1.2
# }

settings = {
    "class_legend_font_size": 16,  ## Controls the size of the top right side legend
    "class_legend_marker_size": 2,
    "annotation_legend_font_size": 10,
    "annotation_background_separation": 0.03,
    "annotation_background_offset": 0,
    "annotation_background_width": 0.1,
    "annotation_background_alpha": 0.1,
    "annotation_font_size": 10,
    "annotation_font_stretch": 0,
    "start_rotation": 90,
    "total_plotted_degrees": 345,
    "internal_labels_rotation": 345,
    "branch_thickness": 3,
    "branch_bracket_depth": 0.8,
    "branch_bracket_width": 0.5,
    "clade_separation": 0.5,
    "clade_marker_size": 70,
    "clade_marker_edge_color": "#555555",
    "clade_marker_edge_width": 1.2,
}

annot_txt = [i + "\t" + str(j) for i, j in settings.items()]

In [19]:
color_higher_taxonomy = [
    ("Clostridia_258483", "olivedrab"),
    ("Lachnospiraceae", "olivedrab"),
    ("Ruminococcaceae", "olivedrab"),
    # ('Oscillospiraceae', 'olivedrab'),
    ("Acutalibacteraceae", "olivedrab"),
    ("Lachnospiraceae", "olivedrab"),
    ("Butyricicoccaceae", "olivedrab"),
    ("Bacilli", "turquoise"),
    ("Coriobacteriia", "teal"),
    ("Actinomycetia", "slateblue"),
    ("Bacteroidia", "lightslategrey"),
    ("Lentisphaeria", "hotpink"),
    ("Verrucomicrobiae", "deeppink"),
    ("Desulfovibrionia", "crimson"),
    ("Negativicutes", "indigo"),
    # ('Peptococcia', 'black')
]

In [20]:
def remove_num_suffix(text):
    sub_text = text.split("_")[-1]
    if sub_text.isdigit():
        return text.split("_")[0]
    return text

In [21]:
counter_for_class_legend = 1

for i in color_higher_taxonomy:
    annot_txt.append("\t".join([i[0], "clade_marker_size", "100"]))
    annot_txt.append("\t".join([i[0], "clade_marker_color", i[1]]))
    annot_txt.append("\t".join([i[0], "clade_marker_edge_color", i[1]]))
    annot_txt.append("\t".join([i[0], "clade_marker_shape", "h"]))

    if i[0] not in [
        "Negativicutes",
        "Desulfovibrionia",
        "Verrucomicrobiae",
        "Lentisphaeria",
        "Actinomycetia",
    ]:
        annot_txt.append("\t".join([i[0], "annotation", remove_num_suffix(i[0])]))
    annot_txt.append("\t".join([i[0], "annotation_background_color", i[1]]))

    if not i[0].endswith("eae"):
        annot_txt.append(
            "\t".join(
                [
                    f"{counter_for_class_legend}. {remove_num_suffix(i[0])}",
                    "clade_marker_size",
                    "50",
                ]
            )
        )
        annot_txt.append(
            "\t".join(
                [
                    f"{counter_for_class_legend}. {remove_num_suffix(i[0])}",
                    "clade_marker_color",
                    i[1],
                ]
            )
        )
        annot_txt.append(
            "\t".join(
                [
                    f"{counter_for_class_legend}. {remove_num_suffix(i[0])}",
                    "clade_marker_shape",
                    "h",
                ]
            )
        )

        counter_for_class_legend += 1

annot_txt.append(
    "\t".join(["Bacteria", "clade_marker_size", "0"])
)  # Hide the clade marker for Bacteria

In [22]:
# annot_txt.append("\t".join(["Negativicutes", "annotation_rotation", "90"]))  # Rotate label 90 degree
# annot_txt.append("\t".join(["Negativicutes", "annotation_background_offset", "10"]))  # Rotate label 90 degree

In [23]:
featsImp.set_index("guide_text")[select_target_cols].sum(axis=1)

guide_text
Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Eubacterium_J.Eubacterium_J plexicaudatum      2.521874
Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Marvinbryantia.Marvinbryantia sp900066075      2.182055
Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.CAG-194.CAG-194 sp000432915                    2.414524
Bacteria.Firmicutes_A.Clostridia_258483.TANB77.CAG-508.CAG-269.CAG-269 sp000431335                                    2.022627
Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Lachnospira                                    1.288780
                                                                                                                        ...   
Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Acutalibacteraceae.Eubacterium_R.Eubacterium_R faecavium      0.462123
Bacteria.Bacteroidota.Bacteroidia.Bacteroidales.Bacteroidaceae.Prevotella.Prevotella sp003447235    

#### Selecting microbes to label

In [24]:
### Option 1: Based on sum of scaled feature importances across all target columns
# selected_legend_label_microbes = featsImp.set_index("guide_text")[select_target_cols].sum(axis=1).sort_values(ascending=False)[:45].index

### Option 2: Hand picked microbes

selected_legend_label_microbes = [
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Eubacterium_J.Eubacterium_J plexicaudatum",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Bariatricus.Coprococcus comes",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Marvinbryantia.Marvinbryantia sp900066075",
    #'Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Ruminococcaceae.Phocea',
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.CAG-194.CAG-194 sp000432915",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Butyribacter.Butyribacter sp001916135",
    "Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Ruminococcaceae.Negativibacillus.Negativibacillus massiliensis",
    "Bacteria.Firmicutes_D.Bacilli.Erysipelotrichales.Coprobacillaceae.Erysipelatoclostridium.Clostridium spiroforme",
    "Bacteria.Firmicutes_A.Clostridia_258483.Clostridiales.Clostridiaceae_222000.Clostridium_T.Clostridium_T chartatabidum",
    "Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Oscillospiraceae_88309.Dysosmobacter.Dysosmobacter welbionis",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Mediterraneibacter_A_155507.Mediterraneibacter_A_155507 faecis",
    #'Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Lachnoclostridium_B.Lachnoclostridium_B sp000765215',
    "Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Oscillospiraceae_88309.Lawsonibacter.Lawsonibacter asaccharolyticus",
    "Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Ruminococcaceae.Massilioclostridium.Massilioclostridium coli",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Lachnospira.Lachnospira eligens",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Mediterraneibacter_A_155507.Mediterraneibacter_A_155507 torques",
    #'Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Copromonas.Copromonas sp000435795',
    "Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Butyricicoccaceae.Butyricicoccus_A_77030.Butyricicoccus_A_77030 sp900604335",
    "Bacteria.Firmicutes_D.Bacilli.Lactobacillales.Lactobacillaceae.Latilactobacillus.Latilactobacillus curvatus",
    "Bacteria.Firmicutes_D.Bacilli.Staphylococcales.Gemellaceae.Gemella.Gemella morbillorum",
    "Bacteria.Desulfobacterota_I.Desulfovibrionia.Desulfovibrionales.Desulfovibrionaceae.Bilophila.Bilophila wadsworthia",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Blautia_A_141781.Blautia_A_141781 hydrogenotrophica",
    "Bacteria.Firmicutes_C.Negativicutes.Veillonellales.Dialisteraceae.Allisonella.Allisonella histaminiformans",
    # "Bacteria.Firmicutes_D.Bacilli.Staphylococcales.Staphylococcaceae.Staphylococcus.Staphylococcus equorum",
    "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Dorea_A.Dorea_A longicatena",
    #'Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Eubacterium_I',
    "Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Acutalibacteraceae.Hydrogeniiclostridium",
    # "Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Butyribacter.Butyribacter intestini",
    # "Bacteria.Firmicutes_A.Clostridia_258483.Clostridiales.Clostridiaceae_222000.Clostridium_T.Clostridium_T paraputrificum_207370",
    "Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Butyricicoccaceae.Agathobaculum.Agathobaculum butyriciproducens",
]

print(len(selected_legend_label_microbes))
selected_legend_label_microbes

23


['Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Eubacterium_J.Eubacterium_J plexicaudatum',
 'Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Bariatricus.Coprococcus comes',
 'Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Marvinbryantia.Marvinbryantia sp900066075',
 'Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.CAG-194.CAG-194 sp000432915',
 'Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Butyribacter.Butyribacter sp001916135',
 'Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Ruminococcaceae.Negativibacillus.Negativibacillus massiliensis',
 'Bacteria.Firmicutes_D.Bacilli.Erysipelotrichales.Coprobacillaceae.Erysipelatoclostridium.Clostridium spiroforme',
 'Bacteria.Firmicutes_A.Clostridia_258483.Clostridiales.Clostridiaceae_222000.Clostridium_T.Clostridium_T chartatabidum',
 'Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Oscillospiraceae_88309.Dys

In [25]:
featsImp[featsImp.index.str.contains("intestini")]

,HEI,vegetables_fruits,oils_nuts_fg_eaten,coffee_fg_eaten,meat_fg_eaten,prevalence,guide_text


In [26]:
def selected_species_renderer(text):
    if "CAG" in text:
        for part in text.split("."):
            if "CAG" not in part:
                mod_text = part
            else:
                return "*:" + mod_text + " uncl."

    elif text.split(" ")[-1].startswith("sp"):
        return "*:" + text.split(".")[-1].split(" ")[0] + " sp."

    return "*:*"


for bac in selected_legend_label_microbes:
    print(bac, "\t", selected_species_renderer(bac), "\n")

Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Eubacterium_J.Eubacterium_J plexicaudatum 	 *:* 

Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Bariatricus.Coprococcus comes 	 *:* 

Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Marvinbryantia.Marvinbryantia sp900066075 	 *:Marvinbryantia sp. 

Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.CAG-194.CAG-194 sp000432915 	 *:Lachnospiraceae uncl. 

Bacteria.Firmicutes_A.Clostridia_258483.Lachnospirales.Lachnospiraceae.Butyribacter.Butyribacter sp001916135 	 *:Butyribacter sp. 

Bacteria.Firmicutes_A.Clostridia_258483.Oscillospirales.Ruminococcaceae.Negativibacillus.Negativibacillus massiliensis 	 *:* 

Bacteria.Firmicutes_D.Bacilli.Erysipelotrichales.Coprobacillaceae.Erysipelatoclostridium.Clostridium spiroforme 	 *:Clostridium sp. 

Bacteria.Firmicutes_A.Clostridia_258483.Clostridiales.Clostridiaceae_222000.Clostridium_T.Clostridium_T chartata

In [27]:
for bac in selected_legend_label_microbes:
    annot_txt.append("\t".join([bac, "annotation", selected_species_renderer(bac)]))
    annot_txt.append("\t".join([bac, "annotation_font_size", "12"]))
    annot_txt.append("\t".join([bac, "clade_marker_color", "black"]))
    annot_txt.append("\t".join([bac, "clade_marker_size", "120"]))
    annot_txt.append("\t".join([bac, "clade_marker_edge_color", "black"]))
    annot_txt.append("\t".join([bac, "clade_marker_shape", "h"]))
    annot_txt.append("\t".join([bac, "annotation_background_color", "black"]))

In [28]:
# annot_txt.append("\t".join(["ring_internal_separator_thickness", "1", "0.5"]))
annot_txt.append("\t".join(["ring_internal_separator_thickness", "2", "0.5"]))
annot_txt.append("\t".join(["ring_internal_separator_thickness", "3", "0.5"]))
annot_txt.append("\t".join(["ring_internal_separator_thickness", "4", "0.5"]))
annot_txt.append("\t".join(["ring_internal_separator_thickness", "5", "0.5"]))
annot_txt.append("\t".join(["ring_internal_separator_thickness", "6", "0.5"]))
annot_txt.append("\t".join(["ring_internal_separator_thickness", "7", "0.5"]))

ring_width = "0.5"
ring_height = "0.6"

annot_txt.append("\t".join(["ring_width", "1", ring_width]))
annot_txt.append("\t".join(["ring_height", "1", ring_height]))

annot_txt.append("\t".join(["ring_label", "2", "HEI"]))
annot_txt.append("\t".join(["ring_width", "2", ring_width]))
annot_txt.append("\t".join(["ring_height", "2", ring_height]))
annot_txt.append("\t".join(["ring_label_color", "2", "darkorange"]))

annot_txt.append("\t".join(["ring_label", "3", "Vegetables-Fruits"]))
annot_txt.append("\t".join(["ring_width", "3", ring_width]))
annot_txt.append("\t".join(["ring_height", "3", ring_height]))
annot_txt.append("\t".join(["ring_label_color", "3", "green"]))

annot_txt.append("\t".join(["ring_label", "4", "Oils-Nuts"]))
annot_txt.append("\t".join(["ring_width", "4", ring_width]))
annot_txt.append("\t".join(["ring_height", "4", ring_height]))
annot_txt.append("\t".join(["ring_label_color", "4", "indigo"]))

annot_txt.append("\t".join(["ring_label", "5", "Coffee"]))
annot_txt.append("\t".join(["ring_width", "5", ring_width]))
annot_txt.append("\t".join(["ring_height", "5", ring_height]))
annot_txt.append("\t".join(["ring_label_color", "5", "saddlebrown"]))

annot_txt.append("\t".join(["ring_label", "6", "Meat"]))
annot_txt.append("\t".join(["ring_width", "6", ring_width]))
annot_txt.append("\t".join(["ring_height", "6", ring_height]))
annot_txt.append("\t".join(["ring_label_color", "6", "red"]))

annot_txt.append("\t".join(["ring_label", "7", "Prevalence"]))
annot_txt.append("\t".join(["ring_width", "7", ring_width]))
annot_txt.append("\t".join(["ring_height", "7", "1"]))
annot_txt.append("\t".join(["ring_label_color", "7", "darkcyan"]))

In [29]:
for e, row in featsImp.iterrows():
    if len(row["guide_text"].split(".")) > 5:
        annot_txt.append("\t".join([row["guide_text"], "ring_shape", "1", "v"]))
        annot_txt.append("\t".join([row["guide_text"], "ring_color", "1", "dimgrey"]))
        annot_txt.append(
            "\t".join([row["guide_text"], "ring_edge_color", "1", "black"])
        )
        annot_txt.append("\t".join([row["guide_text"], "ring_edge_width", "1", "1"]))

        annot_txt.append(
            "\t".join([row["guide_text"], "ring_alpha", "2", str(row["HEI"])])
        )
        annot_txt.append(
            "\t".join([row["guide_text"], "ring_color", "2", "darkorange"])
        )
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_color", "2", "grey"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_width", "2", "0.6"]))

        annot_txt.append(
            "\t".join(
                [row["guide_text"], "ring_alpha", "3", str(row["vegetables_fruits"])]
            )
        )
        annot_txt.append("\t".join([row["guide_text"], "ring_color", "3", "green"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_color", "3", "grey"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_width", "3", "0.6"]))

        annot_txt.append(
            "\t".join(
                [row["guide_text"], "ring_alpha", "4", str(row["oils_nuts_fg_eaten"])]
            )
        )
        annot_txt.append("\t".join([row["guide_text"], "ring_color", "4", "indigo"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_color", "4", "grey"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_width", "4", "0.6"]))

        annot_txt.append(
            "\t".join(
                [row["guide_text"], "ring_alpha", "5", str(row["coffee_fg_eaten"])]
            )
        )
        annot_txt.append(
            "\t".join([row["guide_text"], "ring_color", "5", "saddlebrown"])
        )
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_color", "5", "grey"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_width", "5", "0.6"]))

        annot_txt.append(
            "\t".join([row["guide_text"], "ring_alpha", "6", str(row["meat_fg_eaten"])])
        )
        annot_txt.append("\t".join([row["guide_text"], "ring_color", "6", "red"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_color", "6", "grey"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_width", "6", "0.6"]))

        annot_txt.append(
            "\t".join([row["guide_text"], "ring_alpha", "7", str(row["prevalence"])])
        )
        annot_txt.append("\t".join([row["guide_text"], "ring_color", "7", "darkcyan"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_color", "7", "grey"]))
        # annot_txt.append("\t".join([row['guide_text'], "ring_edge_width", "7", "0.6"]))

In [30]:
with open("nutrition_features_annotations.txt", "w") as f:
    for item in annot_txt:
        f.write("%s\n" % item)

In [31]:
featsImp.columns

Index(['HEI', 'vegetables_fruits', 'oils_nuts_fg_eaten', 'coffee_fg_eaten',
       'meat_fg_eaten', 'prevalence', 'guide_text'],
      dtype='object')